# Ligue 1 Analytics — Data Cleaning Pipeline

This notebook turns the raw WhoScored event-level CSV (`data/raw/ligue1_match_events_ml.csv`) into the clean, relational tables used by the rest of the project (Supabase import, future ML models, website).

Tables produced, all exported to `data/processed/`: `teams.csv` (`Football_Team`), `players.csv` (`Players`, enriched with Transfermarkt data at the end of this notebook), `team_player.csv` (`Team_Player`), `matches.csv` (`Football_Match`, scraped via `soccerdata`: WhoScored schedule + FBref for the matchday/week), `player_match_stats.csv` (`Player_Match_Stats`, goalkeepers included in the same table), and `events.csv` (`Events`, a cleaned version of the raw event log, kept row-per-event, with `qualifiers` stored as valid JSON for JSONB import).

High-level pipeline: build the simple lookup tables (`Players`, `Football_Team`, `Team_Player`) directly from the raw CSV, scrape and merge match schedule/score/week data to build `Football_Match`, explore the `type` and `qualifiers` columns to understand what can be aggregated and build `Player_Match_Stats`, export everything to CSV, fix the `event_id` non-uniqueness issue found in `Events`, convert `qualifiers` to valid JSON for JSONB storage, and finally enrich `Players` with Transfermarkt data (birth date, position, market value) matched by name, replacing `players.csv` with this enriched version.


In [3]:
import pandas as pd

In [4]:
df_raw = pd.read_csv("ligue1_match_events_ml.csv")

In [5]:
df_raw.columns

Index(['game_id', 'period', 'minute', 'second', 'expanded_minute', 'type',
       'outcome_type', 'team_id', 'team', 'player_id', 'player', 'x', 'y',
       'end_x', 'end_y', 'goal_mouth_y', 'goal_mouth_z', 'blocked_x',
       'blocked_y', 'qualifiers', 'is_touch', 'is_shot', 'is_goal',
       'card_type', 'event_id', 'related_event_id', 'related_player_id',
       'match_id'],
      dtype='object')

In [6]:
df_raw.head()

,game_id,period,minute,second,expanded_minute,type,outcome_type,team_id,team,player_id,...,blocked_y,qualifiers,is_touch,is_shot,is_goal,card_type,event_id,related_event_id,related_player_id,match_id
0,1911273,FirstHalf,0,0.0,0,Start,Successful,249,Marseille,NaN,...,NaN,[],False,NaN,NaN,NaN,2,NaN,NaN,1911273
1,1911273,FirstHalf,0,0.0,0,Start,Successful,313,Rennes,NaN,...,NaN,[],False,NaN,NaN,NaN,2,NaN,NaN,1911273
2,1911273,FirstHalf,0,0.0,0,Pass,Successful,249,Marseille,348654.0,...,NaN,"[{'type': {'displayName': 'PassEndY', 'value':...",True,NaN,NaN,NaN,3,NaN,NaN,1911273
3,1911273,FirstHalf,0,4.0,0,Pass,Successful,249,Marseille,362788.0,...,NaN,"[{'type': {'displayName': 'PassEndX', 'value':...",True,NaN,NaN,NaN,4,NaN,NaN,1911273
4,1911273,FirstHalf,0,6.0,0,Pass,Successful,249,Marseille,425574.0,...,NaN,"[{'type': {'displayName': 'Zone', 'value': 56}...",True,NaN,NaN,NaN,5,NaN,NaN,1911273


### Creating the Players table


In [16]:
df_players = df_raw[["player_id", "player"]].drop_duplicates()

In [18]:
df_players.head()


,player_id,player
0,NaN,NaN
2,348654.0,Amine Gouiri
3,362788.0,Leonardo Balerdi
4,425574.0,CJ Egan-Riley
5,338497.0,Angel Gomes


Let's remove the rows with missing (NaN) values.


In [19]:
df_players = df_players.dropna()

Now let's run some checks to verify that the table is correct.


In [20]:
df_players.shape

(551, 2)

In [21]:
df_players["player_id"].nunique()

551

The two values match (551 == 551), so the table was built correctly.


### Now let's create the Football_Team table


In [12]:
df_team = df_raw[["team_id", "team"]].drop_duplicates().dropna()

In [13]:
print(df_team.shape)

(18, 2)


In [94]:
df_team.head(18)

,team_id,team
0,249,Marseille
1,313,Rennes
1583,228,Lyon
1584,309,Lens
3086,248,Monaco
3087,217,Le Havre
4692,246,Toulouse
4693,613,Nice
6156,2832,Paris FC
6157,614,Angers


### Now let's create the Team_Player table


In [181]:
df_team_player = df_raw[["team_id", "player_id"]].drop_duplicates().dropna()

In [30]:
print(df_team_player.shape)


(561, 2)


As we can see, we now have 561 rows compared to the 551 rows in the Players table. This means 10 players moved from one Ligue 1 team to another during the season (mid-season transfers).


### Now let's create the Football_Match table


Our raw dataset is missing some information we need: the precise date, score, and matchday (week) of each match. So we need to scrape two websites to get this information. We'll use the `soccerdata` library (WhoScored for the match schedule, FBref for the matchday/week).


In [210]:
import soccerdata as sd

In [232]:
ws = sd.WhoScored(leagues="FRA-Ligue 1", seasons="25-26")
schedule = ws.read_schedule()

[08/06/26 15:02:48] INFO     Saving cached data to C:\Users\lilia\soccerdata\data\WhoScored          ]8;id=12666414;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12666415;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[08/06/26 15:02:49] INFO     Retrieving calendar for FRA-Ligue 1 2526                              ]8;id=12666420;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666421;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#368\368]8;;\

                    INFO     [1/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666426;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666427;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

                    INFO     [2/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666432;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666433;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

                    INFO     [3/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666438;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666439;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

...

                    INFO     [6/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666456;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666457;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

                    INFO     [7/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666462;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666463;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

                    INFO     [8/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666468;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666469;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

                    INFO     [9/10] Retrieving fixtures for FRA-Ligue 1 2526                       ]8;id=12666474;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666475;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

                    INFO     [10/10] Retrieving fixtures for FRA-Ligue 1 2526                      ]8;id=12666480;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py\whoscored.py]8;;\:]8;id=12666481;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\whoscored.py#399\399]8;;\

In [233]:
schedule.head()


stage_id  game_id  status  \
league      season game                                                     
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille     24609  1911273       6   
                   2025-08-16 Lens-Lyon            24609  1911284       6   
                   2025-08-16 Monaco-Le Havre      24609  1911290       6   
                   2025-08-16 Nice-Toulouse        24609  1911296       6   
                   2025-08-17 Angers-Paris FC      24609  1911275       6   

                                                         start_time  \
league      season game                                               
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  2025-08-15T19:45:00   
                   2025-08-16 Lens-Lyon         2025-08-16T16:00:00   
                   2025-08-16 Monaco-Le Havre   2025-08-16T18:00:00   
                   2025-08-16 Nice-Toulouse     2025-08-16T20:05:00   
                   2025-08-17 Angers-Paris FC   2025-08-17T16:15:00   

                                                home_team_id home_team  \
league      season game                                                  
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille           313    Rennes   
                   2025-08-16 Lens-Lyon                  309      Lens   
                   2025-08-16 Monaco-Le Havre            248    Monaco   
                   2025-08-16 Nice-Toulouse              613      Nice   
                   2025-08-17 Angers-Paris FC            614    Angers   

                                                home_yellow_cards  \
league      season game                                             
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille                  2   
                   2025-08-16 Lens-Lyon                         2   
                   2025-08-16 Monaco-Le Havre                   0   
                   2025-08-16 Nice-Toulouse                     2   
                   2025-08-17 Angers-Paris FC                   2   

                                                home_red_cards  away_team_id  \
league      season game                                                        
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille               1           249   
                   2025-08-16 Lens-Lyon                      0           228   
                   2025-08-16 Monaco-Le Havre                0           217   
                   2025-08-16 Nice-Toulouse                  0           246   
                   2025-08-17 Angers-Paris FC                1          2832   

                                                away_team  ...  period  \
league      season game                                    ...           
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  Marseille  ...       7   
                   2025-08-16 Lens-Lyon              Lyon  ...       7   
                   2025-08-16 Monaco-Le Havre    Le Havre  ...       7   
                   2025-08-16 Nice-Toulouse      Toulouse  ...       7   
                   2025-08-17 Angers-Paris FC    Paris FC  ...       7   

                                                extra_result_field  \
league      season game                                              
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille                None   
                   2025-08-16 Lens-Lyon                       None   
                   2025-08-16 Monaco-Le Havre                 None   
                   2025-08-16 Nice-Toulouse                   None   
                   2025-08-17 Angers-Paris FC                 None   

                                                home_extratime_score  \
league      season game                                                
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille                  None   
                   2025-08-16 Lens-Lyon                         None   
                   2025-08-16 Monaco-Le Havre                   None   
                   2025-08-16 Nice-Toulouse                     None   
    

We still need the matchday (week) for every match, so let's scrape another website and merge the two datasets.


In [234]:
fbref = sd.FBref("FRA-Ligue 1", "2025-2026")
fbref_schedule = fbref.read_schedule()

[08/06/26 15:02:53] INFO     Saving cached data to C:\Users\lilia\soccerdata\data\FBref              ]8;id=12666486;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12666487;file://C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[08/06/26 15:02:54] WARNING  C:\Users\lilia\anaconda3\Lib\site-packages\soccerdata\fbref.py:171:    ]8;id=12666492;file://C:\Users\lilia\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=12666493;file://C:\Users\lilia\anaconda3\Lib\warnings.py#110\110]8;;\
                             FutureWarning: The behavior of DataFrame concatenation with empty or                  
                             all-NA entries is deprecated. In a future version, this will no longer                
                             exclude empty or all-NA columns when determining the result dtypes. To                
                             retain the old behavior, exclude the relevant entries before the                      
                             concat operation.                                                                     
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [235]:
fbref_schedule.columns
fbref_schedule.head()

round  week  day       date  \
league      season game                                                         
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  Ligue 1     1  Fri 2025-08-15   
                   2025-08-16 Lens-Lyon         Ligue 1     1  Sat 2025-08-16   
                   2025-08-16 Monaco-Le Havre   Ligue 1     1  Sat 2025-08-16   
                   2025-08-16 Nice-Toulouse     Ligue 1     1  Sat 2025-08-16   
                   2025-08-17 Angers-Paris FC   Ligue 1     1  Sun 2025-08-17   

                                                 time home_team score  \
league      season game                                                 
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  20:45    Rennes   1–0   
                   2025-08-16 Lens-Lyon         17:00      Lens   0–1   
                   2025-08-16 Monaco-Le Havre   19:00    Monaco   3–1   
                   2025-08-16 Nice-Toulouse     21:05      Nice   0–1   
                   2025-08-17 Angers-Paris FC   17:15    Angers   1–0   

                                                away_team  attendance  \
league      season game                                                 
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  Marseille       28360   
                   2025-08-16 Lens-Lyon              Lyon       38137   
                   2025-08-16 Monaco-Le Havre    Le Havre       11734   
                   2025-08-16 Nice-Toulouse      Toulouse       23212   
                   2025-08-17 Angers-Paris FC    Paris FC       12168   

                                                                 venue  \
league      season game                                                  
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille            Roazhon Park   
                   2025-08-16 Lens-Lyon         Stade Bollaert-Delelis   
                   2025-08-16 Monaco-Le Havre           Stade Louis II   
                   2025-08-16 Nice-Toulouse            Allianz Riviera   
                   2025-08-17 Angers-Paris FC       Stade Raymond Kopa   

                                                          referee  \
league      season game                                             
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille    Jérémie Pignard   
                   2025-08-16 Lens-Lyon          Romain Lissorgue   
                   2025-08-16 Monaco-Le Havre        Ruddy Buquet   
                   2025-08-16 Nice-Toulouse        Thomas Léonard   
                   2025-08-17 Angers-Paris FC   François Letexier   

                                                                                     match_report  \
league      season game                                                                             
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  /en/matches/fbd5fc84/Rennes-Marseille-August-1...   
                   2025-08-16 Lens-Lyon         /en/matches/5959d3ad/Lens-Lyon-August-16-2025-...   
                   2025-08-16 Monaco-Le Havre   /en/matches/772d8629/Monaco-Le-Havre-August-16...   
                   2025-08-16 Nice-Toulouse     /en/matches/086666a1/Nice-Toulouse-August-16-2...   
                   2025-08-17 Angers-Paris FC   /en/matches/c69996e3/Angers-Paris-FC-August-17...   

                                               notes   game_id  
league      season game                                         
FRA-Ligue 1 2526   2025-08-15 Rennes-Marseille  <NA>  fbd5fc84  
                   2025-08-16 Lens-Lyon         <NA>  5959d3ad  
                   2025-08-16 Monaco-Le Havre   <NA>  772d8629  
                   2025-08-16 Nice-Toulouse     <NA>  086666a1  
                   2025-08-17 Angers-Paris FC   <NA>  c69996e3

After a first merge attempt, we lost 34 matches. This was caused by a naming mismatch between the two websites: one used "PSG" and the other "Paris Saint-Germain". We fixed it by normalizing the team name on the WhoScored side to match FBref's full name before merging again.


In [236]:
fbref_schedule = fbref_schedule.reset_index()
fbref_schedule['game'] = fbref_schedule['game'].str.replace("PSG", "Paris Saint-Germain", regex=False)
fbref_schedule = fbref_schedule.set_index(['league', 'season', 'game'])

In [237]:
df_merged = pd.merge(schedule, fbref_schedule, on="game")
df_merged.shape

(306, 57)

Now let's clean this dataset a bit (drop redundant columns, fix the date/timezone, etc.).


In [238]:
df_merged = df_merged[["game_id_x","home_team_id","away_team_id","score","round","week","day","time","date_x","home_score","away_score"]].reset_index()
df_merged = df_merged.rename(columns = {"game_id_x" : "game_id"})
df_merged = df_merged.set_index("game_id")

In [239]:
df_merged.columns
df_merged = df_merged.drop("score", axis=1)
df_merged['match_date'] = df_merged['date_x'].dt.tz_convert('Europe/Paris')
df_merged['match_date'] = df_merged['match_date'].dt.date
df_merged = df_merged.drop(columns=['date_x'])
df_merged = df_merged.drop(columns=['game'])

In [240]:
df_merged.head()


,home_team_id,away_team_id,round,week,day,time,home_score,away_score,match_date
game_id,,,,,,,,,
1911273,313,249,Ligue 1,1,Fri,20:45,1,0,2025-08-15
1911284,309,228,Ligue 1,1,Sat,17:00,0,1,2025-08-16
1911290,248,217,Ligue 1,1,Sat,19:00,3,1,2025-08-16
1911296,613,246,Ligue 1,1,Sat,21:05,0,1,2025-08-16
1911275,614,2832,Ligue 1,1,Sun,17:15,1,0,2025-08-17


In [241]:
df_merged.shape

(306, 9)

### Now let's create the Player_Match_Stats table

This is the most involved table of the project. Before writing any aggregation code, we spent a dedicated exploration phase (see the companion notes / chat history) understanding two columns.

`type` (39 distinct event types) tells us directly what happened on an event (e.g. `Goal`, `SavedShot`, `MissedShots`, `ShotOnPost`, `Save`, `Claim`, `KeeperPickup`, `Foul`, `Card`...). `qualifiers` is a stringified list of dicts (parsed with `ast.literal_eval`) holding finer-grained tags on top of `type` (e.g. `KeyPass`, `IntentionalGoalAssist`, `OwnGoal`, `Cross`, `Yellow`/`Red`...).

Key decisions made during that exploration, applied in the cell below: own goals are detected as `type == 'Goal'` with the `OwnGoal` qualifier present (the `team` on that row is the scorer's own team, so own goals are counted separately in `own_goals` rather than mixed into `goals`). Assists use the `IntentionalGoalAssist` qualifier, not `IntentionalAssist`, which is a much broader/looser tag — we verified this against the total number of goals in the season (863), since `IntentionalAssist` alone had more occurrences than there are goals in the whole season, which is impossible for a true assist flag. Key passes use the `KeyPass` or `ShotAssist` qualifier (a pass that led to a shot, whether or not it scored). Shots on target are `type in ['SavedShot', 'Goal']`, off target is `type == 'MissedShots'`, and post is `type == 'ShotOnPost'`. Fouls are logged twice per event (once per team) with opposite `outcome_type`: `Unsuccessful` means the player committed the foul (this is what we count as `fouls_committed`), `Successful` means the player won the foul.

Goalkeepers are identified per `(player_id, match_id)` as anyone with at least one event of type `Claim`, `KeeperPickup`, `Punch`, `KeeperSweeper` or `CrossNotClaimed`. `Save` was deliberately excluded from this identification list: it turned out to also fire on regular outfield defensive actions (an average of ~6.7 distinct "goalkeepers" per match instead of ~2), so it's not reliable for telling goalkeepers apart from outfield players, even though it's still summed as a raw event elsewhere if needed. We chose to keep one single table for all players, goalkeepers included, rather than splitting into `Field_Player_Match_Stats` / `Goalkeeper_Match_Stats`, since goalkeepers also pass, get cards, etc. Goalkeeper-only columns (e.g. `gk_claims`) are simply 0 for outfield players. Finally, `event_id` turned out to not be unique, even within a single match (only ~1921 distinct values for 467k rows) — see the fix applied further down when building the Events table.


In [242]:
import ast

# ============================================================
# Préparation : parser les qualifiers
# ============================================================
df_raw["qualifiers_parsed"] = df_raw["qualifiers"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) and x != "[]" else []
)

def has_qualifier(qlist, display_name):
    return any(q.get("type", {}).get("displayName") == display_name for q in qlist)

# ============================================================
# Flags par ligne d'événement
# ============================================================
df_raw["is_owngoal"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "OwnGoal"))
df_raw["is_assist"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "IntentionalGoalAssist"))
df_raw["is_keypass"] = df_raw["qualifiers_parsed"].apply(
    lambda q: has_qualifier(q, "KeyPass") or has_qualifier(q, "ShotAssist")
)
df_raw["is_cross"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "Cross"))
df_raw["is_longball"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "Longball"))
df_raw["is_offside"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "PlayerCaughtOffside"))
df_raw["is_corner_taken"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "CornerTaken"))
df_raw["is_yellow"] = df_raw["qualifiers_parsed"].apply(lambda q: has_qualifier(q, "Yellow"))
df_raw["is_red"] = df_raw["qualifiers_parsed"].apply(
    lambda q: has_qualifier(q, "Red") or has_qualifier(q, "SecondYellow")
)

df_raw["is_goal_scored"] = (df_raw["type"] == "Goal") & (~df_raw["is_owngoal"])
df_raw["is_shot_on_target"] = df_raw["type"].isin(["SavedShot", "Goal"]) & (~df_raw["is_owngoal"])
df_raw["is_shot_off_target"] = df_raw["type"] == "MissedShots"
df_raw["is_shot_post"] = df_raw["type"] == "ShotOnPost"
df_raw["is_shot_total"] = df_raw["is_shot"] == True

df_raw["is_pass"] = df_raw["type"] == "Pass"
df_raw["is_pass_success"] = (df_raw["type"] == "Pass") & (df_raw["outcome_type"] == "Successful")

df_raw["is_foul_committed"] = (df_raw["type"] == "Foul") & (df_raw["outcome_type"] == "Unsuccessful")

df_raw["is_card_yellow"] = (df_raw["type"] == "Card") & df_raw["is_yellow"]
df_raw["is_card_red"] = (df_raw["type"] == "Card") & df_raw["is_red"]

df_raw["is_gk_claim"] = df_raw["type"] == "Claim"
df_raw["is_gk_pickup"] = df_raw["type"] == "KeeperPickup"
df_raw["is_gk_punch"] = df_raw["type"] == "Punch"
df_raw["is_gk_sweeper"] = df_raw["type"] == "KeeperSweeper"
df_raw["is_gk_cross_not_claimed"] = df_raw["type"] == "CrossNotClaimed"

# ============================================================
# Agrégation par (player_id, match_id)
# ============================================================
df_player_match_stats = df_raw[df_raw["player_id"] > 0].dropna(subset=["player_id"]).groupby(["player_id", "match_id"]).agg(
    team_id=("team_id", "first"),
    goals=("is_goal_scored", "sum"),
    own_goals=("is_owngoal", "sum"),
    assists=("is_assist", "sum"),
    key_passes=("is_keypass", "sum"),
    total_passes=("is_pass", "sum"),
    successful_passes=("is_pass_success", "sum"),
    crosses=("is_cross", "sum"),
    long_balls=("is_longball", "sum"),
    total_shots=("is_shot_total", "sum"),
    shots_on_target=("is_shot_on_target", "sum"),
    shots_off_target=("is_shot_off_target", "sum"),
    shots_post=("is_shot_post", "sum"),
    fouls_committed=("is_foul_committed", "sum"),
    yellow_cards=("is_card_yellow", "sum"),
    red_cards=("is_card_red", "sum"),
    offsides=("is_offside", "sum"),
    corners_taken=("is_corner_taken", "sum"),
    gk_claims=("is_gk_claim", "sum"),
    gk_pickups=("is_gk_pickup", "sum"),
    gk_punches=("is_gk_punch", "sum"),
    gk_sweeper_actions=("is_gk_sweeper", "sum"),
    gk_crosses_not_claimed=("is_gk_cross_not_claimed", "sum"),
).reset_index()

# ============================================================
# goals_conceded : croisement avec df_merged (table match)
# ============================================================
df_player_match_stats = df_player_match_stats.merge(
    df_merged[["home_team_id", "away_team_id", "home_score", "away_score"]],
    left_on="match_id", right_index=True, how="left"
)

df_player_match_stats["goals_conceded"] = df_player_match_stats.apply(
    lambda row: row["away_score"] if row["team_id"] == row["home_team_id"]
    else (row["home_score"] if row["team_id"] == row["away_team_id"] else None),
    axis=1
)

df_player_match_stats = df_player_match_stats.drop(columns=["home_team_id", "away_team_id", "home_score", "away_score"])

print(df_player_match_stats.shape)
df_player_match_stats.head(10)

(9413, 26)


,player_id,match_id,team_id,goals,own_goals,assists,key_passes,total_passes,successful_passes,crosses,...,yellow_cards,red_cards,offsides,corners_taken,gk_claims,gk_pickups,gk_punches,gk_sweeper_actions,gk_crosses_not_claimed,goals_conceded
0,6683.0,1911338,613,0,0,0,0,25,21,0,...,0,0,0,0,0,0,0,0,0,2.0
1,6683.0,1911346,613,0,0,0,0,62,58,0,...,0,0,0,0,0,0,0,0,0,0.0
2,6683.0,1911441,613,0,0,0,0,3,2,0,...,0,0,0,0,0,0,0,0,0,1.0
3,6683.0,1911449,613,0,0,0,0,60,53,0,...,0,0,0,0,0,0,0,0,0,2.0
4,6683.0,1911460,613,0,0,0,0,33,26,0,...,0,0,0,0,0,0,0,0,0,0.0
5,6683.0,1911465,613,0,0,0,0,29,27,0,...,0,0,0,0,0,0,0,0,0,2.0
6,6683.0,1911476,613,0,0,0,0,35,33,0,...,0,0,0,0,0,0,0,0,0,3.0
7,6683.0,1911495,613,0,0,0,0,50,44,0,...,1,0,0,0,0,0,0,0,0,4.0
8,6683.0,1911498,613,0,0,0,0,71,70,0,...,0,0,0,0,0,0,0,0,0,0.0
9,6683.0,1911513,613,0,0,0,0,26,21,0,...,0,0,1,0,0,0,0,0,0,4.0


### Now let's export all tables to CSV files


In [243]:
import os

os.makedirs("processed", exist_ok=True)

df_players.to_csv("processed/players.csv", index=False)
df_team.to_csv("processed/teams.csv", index=False)
df_team_player.to_csv("processed/team_player.csv", index=False)
df_merged.to_csv("processed/matches.csv", index=True)   # index=True car game_id est l'index
df_player_match_stats.to_csv("processed/player_match_stats.csv", index=False)

print("Tous les CSV ont été exportés dans le dossier 'processed/'")

Tous les CSV ont été exportés dans le dossier 'processed/'


### Finally, let's build and export the Events table

`Events` is essentially the cleaned raw dataset: same columns as the original CSV, minus the temporary `is_xxx` boolean flags and the parsed `qualifiers_parsed` column we created above to build `Player_Match_Stats` (those were only useful for the aggregation step; the raw `qualifiers` string column is kept so anything can still be recomputed later, e.g. for a future xG model or shot/pass maps).


In [244]:
temp_columns = [
    "is_owngoal", "is_assist", "is_keypass", "is_cross", "is_longball",
    "is_offside", "is_corner_taken", "is_yellow", "is_red",
    "is_goal_scored", "is_shot_on_target", "is_shot_off_target", "is_shot_post", "is_shot_total",
    "is_pass", "is_pass_success", "is_foul_committed",
    "is_card_yellow", "is_card_red",
    "is_gk_claim", "is_gk_pickup", "is_gk_punch", "is_gk_sweeper", "is_gk_cross_not_claimed",
    "qualifiers_parsed" 
]

df_events = df_raw.drop(columns=[c for c in temp_columns if c in df_raw.columns])

print(df_events.shape)
df_events.columns.tolist()

(467318, 28)


['game_id',
 'period',
 'minute',
 'second',
 'expanded_minute',
 'type',
 'outcome_type',
 'team_id',
 'team',
 'player_id',
 'player',
 'x',
 'y',
 'end_x',
 'end_y',
 'goal_mouth_y',
 'goal_mouth_z',
 'blocked_x',
 'blocked_y',
 'qualifiers',
 'is_touch',
 'is_shot',
 'is_goal',
 'card_type',
 'event_id',
 'related_event_id',
 'related_player_id',
 'match_id']

In [ ]:
df_events.to_csv("processed/events.csv", index=False)
print("events.csv exporté")

## Fixing the `event_id` primary key issue

While designing the SQL schema for `Events`, we found that `event_id` is **not unique** — only ~1921 distinct values for 467,318 rows, and not even unique within a single match (the same `event_id` can appear on two completely unrelated events, even in the same match and period). No reliable pattern (per match, per period...) explains the repetition, so rather than trying to "rescue" `event_id` as a key, we generate a simple technical primary key `event_pk` (a plain row counter). `event_id` and `related_event_id` are kept as-is in the table for reference, but are not guaranteed unique — this is a known limitation, not currently used for anything critical.


In [ ]:
df_events = pd.read_csv("processed/events.csv")

df_events = df_events.reset_index(drop=True)
df_events.insert(0, "event_pk", range(1, len(df_events) + 1))

print(df_events.shape)
print("event_pk unique:", df_events["event_pk"].nunique() == df_events.shape[0])

df_events.to_csv("processed/events.csv", index=False)
print("events.csv re-exported with event_pk")

## Enriching the Players table with Transfermarkt data

`Players` currently only has `player_id` and `player` (name), no birth date, position, or market value. We looked for an existing WhoScored↔Transfermarkt ID crosswalk first (the [Reep register](https://github.com/withqwerty/reep)), but it only covers about 15% of our 551 players (mostly players who also had spells abroad and therefore a well-populated Wikidata page), so it's not a viable standalone solution — it can only be used later as a shortcut for a handful of players.

Instead, we use the [`dcaribou/transfermarkt-datasets`](https://github.com/dcaribou/transfermarkt-datasets) (also on Kaggle), a clean, weekly-refreshed Transfermarkt export, and match players by name. We first load the Transfermarkt `players` and `player_valuations` tables, then filter to players whose current club plays in Ligue 1 (`current_club_domestic_competition_id == "FR1"`). This must be the current club column, not the club-at-valuation-date column: the latter also matches players who used to play in Ligue 1 years ago but have since moved elsewhere, which pulled in about 2000 irrelevant rows on a first attempt. We then keep only the most recent valuation per player (the raw valuations table has one row per historical valuation date, up to about 49 for a single player), merge with our `players` table by exact name match, and resolve name homonyms (two different real players sharing the exact same name, e.g. two "Ousmane Camara") by cross-checking the WhoScored club (via `team_player` + `teams`) against the Transfermarkt current club name. We also drop duplicate rows coming from mid-season transfers in `team_player` (a transferred player would otherwise appear twice, once per club — for `Players` we only need one row per player regardless of club), and remove the `player_id == 0` ghost row (a leftover of a few unidentified-player events in the raw event log, already spotted earlier in `Player_Match_Stats`, that had also leaked into `Players`).

This name-matching approach recovers about 86% of players on the first, exact-name pass. The rest (mostly accents/diacritics or non-Latin names transliterated differently between the two sources, e.g. Korean names) are left as `NaN` for now — a `rapidfuzz`-based fuzzy-matching pass is a natural next step to close that gap further, but is not required to move forward with the project.


In [ ]:
tm_players = pd.read_csv("../data/raw/transfermarkt_players.csv")
tm_valuations = pd.read_csv("../data/raw/transfermarkt_valuations.csv")

print(tm_players.shape)
print(tm_valuations.shape)

In [ ]:
# Keep only players whose CURRENT club plays in Ligue 1
# (not the club at valuation date, which would also pull in former Ligue 1 players)
tm_players_l1 = tm_players[tm_players["current_club_domestic_competition_id"] == "FR1"]
print(tm_players_l1.shape)
print(tm_players_l1["player_id"].nunique())

In [ ]:
# Keep only the most recent valuation per player
tm_valuations_recent = (
    tm_valuations.sort_values("date")
    .groupby("player_id")
    .tail(1)
)
print(tm_valuations_recent.shape)
print(tm_valuations_recent["player_id"].nunique())

In [ ]:
# Merge current-club Ligue 1 players with their most recent valuation
tm_ligue1 = pd.merge(
    tm_players_l1,
    tm_valuations_recent[["player_id", "date", "market_value_in_eur"]],
    on="player_id",
    how="left"
)
print(tm_ligue1.shape)

In [ ]:
# Match with our WhoScored players table, by exact name
players_matched = pd.merge(
    df_players, tm_ligue1,
    left_on="player", right_on="name",
    how="left",
    suffixes=("_ws", "_tm")
)

matched = players_matched["player_id_tm"].notna().sum() if "player_id_tm" in players_matched.columns else players_matched["player_id_y"].notna().sum()
total = players_matched["player"].nunique()
print(f"{matched} / {total} matched on exact name ({matched/total*100:.1f}%)")

### Resolving homonyms

A handful of WhoScored players matched two different Transfermarkt player IDs sharing the exact same name (real homonyms, e.g. two different "Ousmane Camara", one at Angers one at Auxerre). We disambiguate by comparing the WhoScored club (via `team_player` + `teams`) to the Transfermarkt current club name, and keep only the row where the two clubs agree.


In [ ]:
# Bring in the WhoScored club name for each player, to disambiguate homonyms
players_matched = pd.merge(players_matched, df_team_player, on="player_id", how="left", suffixes=("", "_tp"))
players_matched = pd.merge(players_matched, df_team, on="team_id", how="left")

players_matched["club_match"] = players_matched.apply(
    lambda row: str(row["team"]).lower() in str(row["current_club_name"]).lower()
    if pd.notna(row["team"]) and pd.notna(row["current_club_name"])
    else False,
    axis=1
)

dup_ids = players_matched[players_matched.duplicated(subset="player_id", keep=False)]["player_id"].unique()
non_dups = players_matched[~players_matched["player_id"].isin(dup_ids)]
dups_resolved = players_matched[players_matched["player_id"].isin(dup_ids) & players_matched["club_match"]]

players_matched = pd.concat([non_dups, dups_resolved])
print(players_matched.shape)
print("Remaining duplicates on player_id:", players_matched["player_id"].duplicated().sum())

### Final selection, cleanup and export

Select only the columns we want to keep, drop the duplicate rows introduced by `team_player` (mid-season transfers — for `Players` we only need one row per player), remove the `player_id == 0` ghost row, and replace `players.csv` with this enriched version.


In [ ]:
players_enriched = players_matched[[
    "player_id", "player", "date_of_birth", "position", "sub_position", "foot",
    "height_in_cm", "country_of_birth", "country_of_citizenship",
    "market_value_in_eur", "highest_market_value_in_eur", "contract_expiration_date"
]].rename(columns={
    # player_id_tm from Transfermarkt is kept separately below if present
})

if "player_id_tm" in players_matched.columns:
    players_enriched["transfermarkt_id"] = players_matched["player_id_tm"]
elif "player_id_y" in players_matched.columns:
    players_enriched["transfermarkt_id"] = players_matched["player_id_y"]

# One row per player: drop duplicates coming from mid-season transfers in team_player
players_enriched = players_enriched.drop_duplicates(subset="player_id", keep="first")

# Remove the ghost player_id == 0 row
players_enriched = players_enriched[players_enriched["player_id"] > 0]

print(players_enriched.shape)
print("transfermarkt_id missing:", players_enriched["transfermarkt_id"].isna().sum())

players_enriched.to_csv("processed/players.csv", index=False)
print("players.csv replaced with the enriched version")

## Converting qualifiers to valid JSON (for JSONB storage in Supabase)

The `qualifiers` column, as exported so far, is a Python `repr()` string (single quotes, not valid JSON), which Postgres would reject if the column is declared as `JSONB`. We convert it to an actual JSON string with `ast.literal_eval` followed by `json.dumps`, then double-check every row parses with a real JSON parser before re-exporting. The SQL column should be declared as `qualifiers JSONB` rather than `TEXT`.


In [ ]:
df_events = pd.read_csv("processed/events.csv")

import ast
import json

def to_valid_json(qualifiers_str):
    if pd.isna(qualifiers_str) or qualifiers_str == "[]":
        return "[]"
    parsed = ast.literal_eval(qualifiers_str)
    return json.dumps(parsed)

df_events["qualifiers"] = df_events["qualifiers"].apply(to_valid_json)

def is_valid_json(s):
    try:
        json.loads(s)
        return True
    except (json.JSONDecodeError, TypeError):
        return False

invalid_count = (~df_events["qualifiers"].apply(is_valid_json)).sum()
print(df_events.shape)
print("Invalid JSON rows:", invalid_count)

df_events.to_csv("processed/events.csv", index=False)
print("events.csv re-exported with qualifiers as valid JSON")

# We change some columns names and format to import in SQL

In [2]:

for name in ["teams", "players", "team_player", "matches", "player_match_stats", "events"]:
    df = pd.read_csv(f"../data/processed/{name}.csv")
    print(name, "->", df.columns.tolist())
    print()

teams -> ['team_id', 'team']

players -> ['player_id', 'player', 'transfermarkt_id', 'date_of_birth', 'position', 'sub_position', 'foot', 'height_in_cm', 'country_of_birth', 'country_of_citizenship', 'market_value_in_eur', 'highest_market_value_in_eur', 'contract_expiration_date']

team_player -> ['team_id', 'player_id']

matches -> ['game_id', 'home_team_id', 'away_team_id', 'round', 'week', 'day', 'time', 'home_score', 'away_score', 'match_date']

player_match_stats -> ['player_id', 'match_id', 'team_id', 'goals', 'own_goals', 'assists', 'key_passes', 'total_passes', 'successful_passes', 'crosses', 'long_balls', 'total_shots', 'shots_on_target', 'shots_off_target', 'shots_post', 'fouls_committed', 'yellow_cards', 'red_cards', 'offsides', 'corners_taken', 'gk_claims', 'gk_pickups', 'gk_punches', 'gk_sweeper_actions', 'gk_crosses_not_claimed', 'goals_conceded']

events -> ['event_pk', 'game_id', 'period', 'minute', 'second', 'expanded_minute', 'type', 'outcome_type', 'team_id', 'team',

In [4]:

# 1. teams.csv
df_team = pd.read_csv("../data/processed/teams.csv")
df_team = df_team.rename(columns={"team": "team_name"})
df_team.to_csv("../data/processed/teams.csv", index=False)

# 2. players.csv
df_players = pd.read_csv("../data/processed/players.csv")
df_players = df_players.rename(columns={"player": "player_name"})
df_players.to_csv("../data/processed/players.csv", index=False)

# 3. matches.csv
df_matches = pd.read_csv("../data/processed/matches.csv")
df_matches = df_matches.drop(columns=["round", "day", "time"])
df_matches = df_matches.rename(columns={"week": "matchday"})
df_matches.to_csv("../data/processed/matches.csv", index=False)

# 4. events.csv
df_events = pd.read_csv("../data/processed/events.csv")
df_events = df_events.drop(columns=["game_id"])
df_events.to_csv("../data/processed/events.csv", index=False)

# Vérification finale de toutes les colonnes
print("teams:", df_team.columns.tolist())
print("players:", df_players.columns.tolist())
print("matches:", df_matches.columns.tolist())
print("events:", df_events.columns.tolist())

teams: ['team_id', 'team_name']
players: ['player_id', 'player_name', 'transfermarkt_id', 'date_of_birth', 'position', 'sub_position', 'foot', 'height_in_cm', 'country_of_birth', 'country_of_citizenship', 'market_value_in_eur', 'highest_market_value_in_eur', 'contract_expiration_date']
matches: ['game_id', 'home_team_id', 'away_team_id', 'matchday', 'home_score', 'away_score', 'match_date']
events: ['event_pk', 'period', 'minute', 'second', 'expanded_minute', 'type', 'outcome_type', 'team_id', 'team', 'player_id', 'player', 'x', 'y', 'end_x', 'end_y', 'goal_mouth_y', 'goal_mouth_z', 'blocked_x', 'blocked_y', 'qualifiers', 'is_touch', 'is_shot', 'is_goal', 'card_type', 'event_id', 'related_event_id', 'related_player_id', 'match_id']


In [6]:
df_players = pd.read_csv("../data/processed/players.csv")
print(df_players.dtypes)

player_id                      float64
player_name                     object
transfermarkt_id               float64
date_of_birth                   object
position                        object
sub_position                    object
foot                            object
height_in_cm                   float64
country_of_birth                object
country_of_citizenship          object
market_value_in_eur            float64
highest_market_value_in_eur    float64
contract_expiration_date        object
dtype: object


In [9]:
int_cols = ["player_id", "transfermarkt_id", "height_in_cm", "market_value_in_eur", "highest_market_value_in_eur"]

for col in int_cols:
    df_players[col] = df_players[col].astype("Int64")

df_players.to_csv("../data/processed/players.csv", index=False)

print(df_players.dtypes)

player_id                       Int64
player_name                    object
transfermarkt_id                Int64
date_of_birth                  object
position                       object
sub_position                   object
foot                           object
height_in_cm                    Int64
country_of_birth               object
country_of_citizenship         object
market_value_in_eur             Int64
highest_market_value_in_eur     Int64
contract_expiration_date       object
dtype: object


In [10]:
df_teams = pd.read_csv("../data/processed/teams.csv")
df_teams["team_id"] = df_teams["team_id"].astype("Int64")
df_teams.to_csv("../data/processed/teams.csv", index=False)

df_matches = pd.read_csv("../data/processed/matches.csv")
for col in ["game_id", "home_team_id", "away_team_id", "matchday", "home_score", "away_score"]:
    df_matches[col] = df_matches[col].astype("Int64")
df_matches.to_csv("../data/processed/matches.csv", index=False)

df_team_player = pd.read_csv("../data/processed/team_player.csv")
for col in ["team_id", "player_id"]:
    df_team_player[col] = df_team_player[col].astype("Int64")
df_team_player.to_csv("../data/processed/team_player.csv", index=False)

df_pms = pd.read_csv("../data/processed/player_match_stats.csv")
int_cols_pms = [
    "player_id", "match_id", "team_id", "goals", "own_goals", "assists", "key_passes",
    "total_passes", "successful_passes", "crosses", "long_balls", "total_shots",
    "shots_on_target", "shots_off_target", "shots_post", "fouls_committed",
    "yellow_cards", "red_cards", "offsides", "corners_taken", "gk_claims",
    "gk_pickups", "gk_punches", "gk_sweeper_actions", "gk_crosses_not_claimed", "goals_conceded"
]
for col in int_cols_pms:
    df_pms[col] = df_pms[col].astype("Int64")
df_pms.to_csv("../data/processed/player_match_stats.csv", index=False)

df_events = pd.read_csv("../data/processed/events.csv")
for col in ["event_pk", "player_id", "team_id", "related_event_id", "related_player_id", "event_id", "match_id"]:
    df_events[col] = df_events[col].astype("Int64")
df_events.to_csv("../data/processed/events.csv", index=False)



teams: {'team_id': Int64Dtype(), 'team_name': dtype('O')}

matches: {'game_id': Int64Dtype(), 'home_team_id': Int64Dtype(), 'away_team_id': Int64Dtype(), 'matchday': Int64Dtype(), 'home_score': Int64Dtype(), 'away_score': Int64Dtype(), 'match_date': dtype('O')}

team_player: {'team_id': Int64Dtype(), 'player_id': Int64Dtype()}

player_match_stats: {'player_id': Int64Dtype(), 'match_id': Int64Dtype(), 'team_id': Int64Dtype(), 'goals': Int64Dtype(), 'own_goals': Int64Dtype(), 'assists': Int64Dtype(), 'key_passes': Int64Dtype(), 'total_passes': Int64Dtype(), 'successful_passes': Int64Dtype(), 'crosses': Int64Dtype(), 'long_balls': Int64Dtype(), 'total_shots': Int64Dtype(), 'shots_on_target': Int64Dtype(), 'shots_off_target': Int64Dtype(), 'shots_post': Int64Dtype(), 'fouls_committed': Int64Dtype(), 'yellow_cards': Int64Dtype(), 'red_cards': Int64Dtype(), 'offsides': Int64Dtype(), 'corners_taken': Int64Dtype(), 'gk_claims': Int64Dtype(), 'gk_pickups': Int64Dtype(), 'gk_punches': Int64Dtype

Avant: (561, 2)
     team_id  player_id
532      146          0
Après: (560, 2)
